<table>
    <tr>
      <td>Tratamiento y Gestión de Datos Masivos - Máster en Inteligencia Artificial - Facultad de Informática - UCM
      </td>
      <td>
      <img src="https://biblioteca.ucm.es/data/cont/media/www/pag-88746//escudo.jpg"  width=50/>
      </td>
     </tr>
</table>

# Práctica: ETL de una Red Social con PySpark
## Pablo C. Cañizares

En esta práctica construirás un pipeline ETL completo para procesar datos de **SocialLab**, una red social tipo Twitter. Partirás de 4 ficheros JSON con datos **intencionalmente sucios** y los transformarás en DataFrames limpios y agregados, siguiendo la arquitectura **Raw → Silver → Gold**.

Los datos presentan problemas reales de calidad: timestamps en 5 formatos distintos, encoding UTF-8 roto, registros duplicados, referencias huérfanas, hashtags inconsistentes y cuentas de spam.

> **Nota:** Este notebook está diseñado para ejecutarse en **Databricks**. La variable `spark` (SparkSession) ya está disponible automáticamente. Sube los 4 ficheros JSON a DBFS antes de empezar.

## Dataset: SocialLab

SocialLab es una red social tipo Twitter con 4 entidades principales:

| Fichero | Descripción | Registros aprox. | Campos principales |
|---|---|---|---|
| `users.json` | Usuarios registrados | ~3.200 | `_id`, `username`, `display_name`, `email`, `bio`, `topic`, `is_spam`, `created_at` |
| `posts.json` | Publicaciones (tweets) | ~80.000 | `_id`, `user_id`, `text`, `hashtags`, `created_at`, `likes_count` |
| `likes.json` | Likes a posts | ~200.000 | `_id`, `user_id`, `post_id`, `created_at` |
| `follows.json` | Relaciones de seguimiento | ~50.000 | `_id`, `follower_id`, `following_id`, `created_at` |

### Problemas de calidad (intencionados)

| Problema | Afecta a | Ejemplo |
|---|---|---|
| 5 formatos de timestamp | Todos | Epoch, ISO, US (`03/29/2026 06:30 PM`), EU (`29-03-2026 18:30`), Relativo (`hace 3 días`) |
| Encoding UTF-8 roto | users, posts | `Ã¡` en vez de `á` |
| Usuarios duplicados | users | Mismo email, distinto `_id` |
| Posts/likes/follows huérfanos | posts, likes, follows | Referencias a usuarios eliminados (`u_ghost_*`) |
| Self-follows | follows | `follower_id == following_id` |
| Likes duplicados | likes | Mismo usuario da like al mismo post 2 veces |
| Hashtags inconsistentes | posts | `#Cloud`, `cloud`, ` #CLOUD.` → deberían ser lo mismo |
| Spam burst | posts | 100 posts idénticos del mismo usuario |
| Campos faltantes | todos | ~4% registros sin campos obligatorios |

> **Instrucciones:** Sube los ficheros `users.json`, `posts.json`, `likes.json` y `follows.json` a Databricks (menú lateral → "Data" → "Upload to DBFS") y ajusta las rutas en la celda de Setup.

## Índice

### Bloque 0: Setup y carga
- [Ejercicio 1: Carga de los JSON](#ejercicio-1)
- [Ejercicio 2: Exploración del schema](#ejercicio-2)
- [Ejercicio 3: Conteo de registros](#ejercicio-3)

### Bloque 1: Diagnóstico de calidad
- [Ejercicio 4: Nulos por columna](#ejercicio-4)
- [Ejercicio 5: Usuarios duplicados](#ejercicio-5)
- [Ejercicio 6: Posts huérfanos](#ejercicio-6)
- [Ejercicio 7: Self-follows](#ejercicio-7)
- [Ejercicio 8: Likes duplicados](#ejercicio-8)
- [Ejercicio 9: Formatos de timestamp](#ejercicio-9)
- [Ejercicio 10: Encoding roto](#ejercicio-10)
- [Ejercicio 11: Hashtags inconsistentes](#ejercicio-11)

### Bloque 2: Limpieza — Capa Silver
- [Ejercicio 12: UDF fix_encoding](#ejercicio-12)
- [Ejercicio 13: UDF parse_timestamp](#ejercicio-13)
- [Ejercicio 14: Aplicar parse_timestamp](#ejercicio-14)
- [Ejercicio 15: Normalizar usernames](#ejercicio-15)
- [Ejercicio 16: UDF normalize_hashtags](#ejercicio-16)
- [Ejercicio 17: Eliminar registros incompletos](#ejercicio-17)
- [Ejercicio 18: Deduplicar usuarios](#ejercicio-18)
- [Ejercicio 19: Eliminar posts huérfanos](#ejercicio-19)
- [Ejercicio 20: Eliminar likes huérfanos](#ejercicio-20)
- [Ejercicio 21: Limpiar follows](#ejercicio-21)
- [Ejercicio 22: Deduplicar likes](#ejercicio-22)
- [Ejercicio 23: Detectar spam](#ejercicio-23)

### Bloque 3: Persistencia en Parquet
- [Ejercicio 24: Escribir Silver como Parquet](#ejercicio-24)
- [Ejercicio 25: Leer y verificar Parquet](#ejercicio-25)
- [Ejercicio 26: Comparar JSON vs Parquet](#ejercicio-26)

### Bloque 4: Agregaciones — Capa Gold
- [Ejercicio 27: Posts por usuario](#ejercicio-27)
- [Ejercicio 28: Likes dados y recibidos](#ejercicio-28)
- [Ejercicio 29: Followers y following](#ejercicio-29)
- [Ejercicio 30: Tabla user_stats unificada](#ejercicio-30)
- [Ejercicio 31: Métricas derivadas](#ejercicio-31)
- [Ejercicio 32: Ranking de posts](#ejercicio-32)
- [Ejercicio 33: Hashtag trends](#ejercicio-33)
- [Ejercicio 34: Ranking diario de hashtags](#ejercicio-34)
- [Ejercicio 35: Features de spam](#ejercicio-35)
- [Ejercicio 36: Aristas de interés compartido](#ejercicio-36)

### Bloque 5: Análisis final
- [Ejercicio 37: Top engagement](#ejercicio-37)
- [Ejercicio 38: Proporción de spam](#ejercicio-38)
- [Ejercicio 39: Top hashtags](#ejercicio-39)
- [Ejercicio 40: Análisis del grafo](#ejercicio-40)

---

## Setup

In [ ]:
# En Databricks la variable `spark` ya existe.
# Importamos las librerías necesarias.
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, ArrayType, TimestampType, IntegerType
from pyspark.sql import Window
from datetime import datetime, timezone, timedelta
import re

# Rutas a los ficheros JSON en DBFS (ajustar si es necesario)
BASE_PATH = "/FileStore/tables/sociallab"
USERS_PATH = f"{BASE_PATH}/users.json"
POSTS_PATH = f"{BASE_PATH}/posts.json"
LIKES_PATH = f"{BASE_PATH}/likes.json"
FOLLOWS_PATH = f"{BASE_PATH}/follows.json"

# Rutas de salida
SILVER_PATH = f"{BASE_PATH}/silver"
GOLD_PATH = f"{BASE_PATH}/gold"

---

# Bloque 0: Setup y carga

---

<a id="ejercicio-1"></a>
### Ejercicio 1: Carga de los JSON
Carga los 4 ficheros JSON como DataFrames de Spark y muestra las primeras 5 filas de cada uno con `display()`.

In [ ]:
# Tu solución aquí

---

<a id="ejercicio-2"></a>
### Ejercicio 2: Exploración del schema
Muestra el schema de cada DataFrame con `printSchema()`. Observa los tipos inferidos.

In [ ]:
# Tu solución aquí

---

<a id="ejercicio-3"></a>
### Ejercicio 3: Conteo de registros
¿Cuántos registros y cuántas columnas tiene cada DataFrame? Muestra el resultado de forma clara.

In [ ]:
# Tu solución aquí

---

# Bloque 1: Diagnóstico de calidad

Antes de limpiar, debemos entender qué problemas tienen nuestros datos. En esta sección explorarás cada tipo de problema presente en el dataset.

---

<a id="ejercicio-4"></a>
### Ejercicio 4: Nulos por columna en users
Cuenta cuántos valores nulos hay en cada columna del DataFrame de usuarios. Usa `agg` con una lista de expresiones.

In [ ]:
# Tu solución aquí

---

<a id="ejercicio-5"></a>
### Ejercicio 5: Usuarios duplicados por email
Detecta cuántos emails aparecen más de una vez. Muestra los 10 emails más repetidos con su frecuencia.

In [ ]:
# Tu solución aquí

---

<a id="ejercicio-6"></a>
### Ejercicio 6: Posts huérfanos
Encuentra posts cuyo `user_id` no existe en la tabla de usuarios. ¿Cuántos hay? Usa un **anti-join** (left join + filtro por nulo).

In [ ]:
# Tu solución aquí

---

<a id="ejercicio-7"></a>
### Ejercicio 7: Self-follows
Encuentra follows donde `follower_id == following_id`. ¿Cuántos hay?

In [ ]:
# Tu solución aquí

---

<a id="ejercicio-8"></a>
### Ejercicio 8: Likes duplicados
Detecta likes donde la combinación (`user_id`, `post_id`) aparece más de una vez. ¿Cuántos pares duplicados existen?

In [ ]:
# Tu solución aquí

---

<a id="ejercicio-9"></a>
### Ejercicio 9: Formatos de timestamp
El campo `created_at` de users tiene 5 formatos distintos. Usa `rlike` (regex) para clasificar cada registro según su formato y cuenta cuántos hay de cada tipo:
1. **Epoch**: solo dígitos, ≥10 caracteres (e.g. `1711741200`)
2. **ISO**: contiene `T` y `+` o `Z` (e.g. `2026-03-29T18:30:00Z`)
3. **US**: patrón `MM/DD/YYYY` (e.g. `03/29/2026 06:30 PM`)
4. **EU**: patrón `DD-MM-YYYY` (e.g. `29-03-2026 18:30`)
5. **Relativo**: empieza por `hace` (e.g. `hace 3 días`)

In [ ]:
# Tu solución aquí

---

<a id="ejercicio-10"></a>
### Ejercicio 10: Encoding roto
Encuentra registros en users cuyo `display_name` o `bio` contiene el carácter `Ã` (señal de encoding UTF-8 doble). ¿Cuántos hay?

In [ ]:
# Tu solución aquí

---

<a id="ejercicio-11"></a>
### Ejercicio 11: Hashtags inconsistentes
Explota el array de hashtags de posts, y muestra que el mismo concepto aparece en múltiples formas. Agrupa por la versión normalizada (lowercase, sin `#`, sin punto final) y cuenta cuántas variantes tiene cada uno.

In [ ]:
# Tu solución aquí

---

# Bloque 2: Limpieza — Capa Silver

En esta sección transformarás los datos sucios en DataFrames limpios. Cada ejercicio aborda un problema de calidad específico. Al final tendrás 4 DataFrames listos para persistir como Parquet.

### Concepto clave: UDFs (User Defined Functions)

Cuando las funciones built-in de Spark no son suficientes, puedes definir tus propias funciones con `@F.udf(TipoRetorno())`. La función recibe valores Python normales y devuelve el resultado transformado. Spark la aplica fila a fila de forma distribuida.

```python
@F.udf(StringType())
def mi_funcion(texto):
    if texto is None:
        return None
    return texto.upper()

df = df.withColumn("columna", mi_funcion(F.col("columna")))
```

---

<a id="ejercicio-12"></a>
### Ejercicio 12: UDF fix_encoding
Crea una UDF que repare el encoding roto reemplazando las secuencias corruptas por los caracteres correctos:
- `Ã¡` → `á`, `Ã©` → `é`, `Ã­` → `í`, `Ã³` → `ó`, `Ãº` → `ú`, `Ã±` → `ñ`

Aplícala a las columnas `display_name` y `bio` de users.

In [ ]:
# Tu solución aquí

---

<a id="ejercicio-13"></a>
### Ejercicio 13: UDF parse_timestamp
Crea una UDF que parsee los 5 formatos de timestamp a `datetime` UTC:
1. **Epoch**: `"1711741200"` → `datetime.fromtimestamp()`
2. **ISO**: `"2026-03-29T18:30:00Z"` → `datetime.fromisoformat()` (reemplazar `Z` por `+00:00`)
3. **US**: `"03/29/2026 06:30 PM"` → `strptime("%m/%d/%Y %I:%M %p")`
4. **EU**: `"29-03-2026 18:30"` → `strptime("%d-%m-%Y %H:%M")`
5. **Relativo**: `"hace 3 días"` → `now() - timedelta(days=3)`

> **Pista:** Intenta cada formato en orden con bloques try/except. Si ninguno funciona, devuelve `None`.

In [ ]:
# Tu solución aquí

---

<a id="ejercicio-14"></a>
### Ejercicio 14: Aplicar parse_timestamp a los 4 DataFrames
Aplica la UDF `parse_timestamp` a la columna `created_at` de los 4 DataFrames. Después, verifica que no quedan timestamps nulos que antes no lo eran.

In [ ]:
# Tu solución aquí

---

<a id="ejercicio-15"></a>
### Ejercicio 15: Normalizar usernames
Convierte todos los usernames a minúsculas y reemplaza espacios por guiones bajos `_`.

In [ ]:
# Tu solución aquí

---

<a id="ejercicio-16"></a>
### Ejercicio 16: UDF normalize_hashtags
Crea una UDF que reciba un array de hashtags y devuelva un array limpio:
- Quitar espacios al inicio/final
- Quitar `#` inicial
- Quitar `.` final
- Convertir a minúsculas
- Eliminar tags con menos de 2 caracteres
- Eliminar duplicados dentro del mismo array

In [ ]:
# Tu solución aquí

---

<a id="ejercicio-17"></a>
### Ejercicio 17: Eliminar registros incompletos
Elimina:
- **Users** sin `_id` o sin `username`
- **Posts** sin `_id`, `text`, `user_id` o `created_at`
- **Likes** sin `_id`, `user_id` o `post_id`
- **Follows** sin `_id`, `follower_id` o `following_id`

Muestra cuántos registros se eliminan de cada DataFrame.

In [ ]:
# Tu solución aquí

---

<a id="ejercicio-18"></a>
### Ejercicio 18: Deduplicar usuarios por email
Usa una **Window function** con `partitionBy("email")` y `orderBy("_id")` para asignar un número de fila. Quédate solo con la primera ocurrencia (row_number == 1).

In [ ]:
# Tu solución aquí

---

<a id="ejercicio-19"></a>
### Ejercicio 19: Eliminar posts huérfanos
Usa un `inner join` entre posts y los `_id` de users válidos para quedarte solo con posts de usuarios existentes.

In [ ]:
# Tu solución aquí

---

<a id="ejercicio-20"></a>
### Ejercicio 20: Eliminar likes huérfanos
Elimina likes que referencien a un `user_id` o `post_id` que no existen en los DataFrames limpios. Necesitas dos joins.

In [ ]:
# Tu solución aquí

---

<a id="ejercicio-21"></a>
### Ejercicio 21: Limpiar follows
Elimina:
1. **Self-follows**: donde `follower_id == following_id`
2. **Ghost follows**: donde `follower_id` o `following_id` no existen como usuarios válidos

Muestra cuántos se eliminan en cada paso.

In [ ]:
# Tu solución aquí

---

<a id="ejercicio-22"></a>
### Ejercicio 22: Deduplicar likes
Elimina likes duplicados (mismo `user_id` + `post_id`), quedándote con el más antiguo (primer `created_at`). Usa una Window function.

In [ ]:
# Tu solución aquí

---

<a id="ejercicio-23"></a>
### Ejercicio 23: Detectar spam
Detecta posts de spam: si un usuario publica el **mismo texto más de 3 veces**, marca esos posts como `is_spam = True`. Además, propaga el flag `is_spam` de la tabla de users a los posts de ese usuario.

> **Pista:** Usa una Window con `partitionBy("user_id", "text")` y `count("*")` para contar repeticiones.

In [ ]:
# Tu solución aquí

---

# Bloque 3: Persistencia en Parquet

### Concepto clave: Formato columnar

Parquet es un formato **columnar** optimizado para consultas analíticas. A diferencia de JSON (orientado a filas), Parquet almacena cada columna de forma contigua, permitiendo:
- **Compresión** mucho más eficiente (datos similares juntos)
- **Lectura selectiva**: si solo necesitas 3 columnas de 20, Spark solo lee esas 3
- **Preservación del schema**: los tipos de datos se mantienen (no hay que inferirlos cada vez)

---

<a id="ejercicio-24"></a>
### Ejercicio 24: Escribir Silver como Parquet
Escribe los 4 DataFrames limpios como ficheros Parquet en la carpeta `silver/`.

In [ ]:
# Tu solución aquí

---

<a id="ejercicio-25"></a>
### Ejercicio 25: Leer y verificar Parquet
Relee los Parquet y verifica que el schema y el número de filas coinciden con los DataFrames originales.

In [ ]:
# Tu solución aquí

---

<a id="ejercicio-26"></a>
### Ejercicio 26: Comparar rendimiento JSON vs Parquet
Mide el tiempo de lectura + `count()` para JSON y Parquet. ¿Cuántas veces más rápido es Parquet?

In [ ]:
# Tu solución aquí

---

# Bloque 4: Agregaciones — Capa Gold

La capa Gold contiene tablas analíticas derivadas de los datos limpios (Silver). Aquí construirás métricas de usuario, rankings de posts, tendencias de hashtags y features para detección de spam.

> **Nota:** A partir de aquí trabajamos con los DataFrames leídos desde Parquet (Silver). Si los tienes en memoria de los ejercicios anteriores, puedes seguir usándolos directamente.

In [ ]:
# Relectura desde Silver (por si se ha reiniciado el cluster)
users = spark.read.parquet(f"{SILVER_PATH}/users")
posts = spark.read.parquet(f"{SILVER_PATH}/posts")
likes = spark.read.parquet(f"{SILVER_PATH}/likes")
follows = spark.read.parquet(f"{SILVER_PATH}/follows")

---

<a id="ejercicio-27"></a>
### Ejercicio 27: Posts por usuario
Cuenta cuántos posts ha publicado cada usuario. Calcula también la fecha de su primer y último post.

In [ ]:
# Tu solución aquí

---

<a id="ejercicio-28"></a>
### Ejercicio 28: Likes dados y recibidos por usuario
Calcula dos métricas:
- **Likes dados**: cuántos likes ha dado cada usuario (`likes.user_id`)
- **Likes recibidos**: cuántos likes han recibido los posts de cada usuario (requiere un join entre likes y posts para obtener el `user_id` del autor)

In [ ]:
# Tu solución aquí

---

<a id="ejercicio-29"></a>
### Ejercicio 29: Followers y following por usuario
Cuenta cuántos seguidores (`followers_count`) y cuántos seguidos (`following_count`) tiene cada usuario.

In [ ]:
# Tu solución aquí

---

<a id="ejercicio-30"></a>
### Ejercicio 30: Tabla user_stats unificada
Une todos los contadores anteriores en una sola tabla `user_stats` usando **left joins** sobre la tabla de users. Rellena los nulos con 0 usando `fillna`.

> **Pista:** La tabla base es `users.select("_id", "username", "display_name", "is_spam", "created_at")`. Haz left join con cada tabla de contadores.

In [ ]:
# Tu solución aquí

---

<a id="ejercicio-31"></a>
### Ejercicio 31: Métricas derivadas
Añade a `user_stats` tres columnas calculadas:
- `avg_likes_per_post` = likes_received / posts_count (0 si no tiene posts)
- `days_active` = días entre primer y último post + 1 (0 si no tiene posts)
- `posts_per_day` = posts_count / days_active (0 si days_active es 0)

In [ ]:
# Tu solución aquí

---

<a id="ejercicio-32"></a>
### Ejercicio 32: Ranking de posts por engagement real
Recalcula los likes de cada post a partir de la tabla de likes (ignorando el campo `likes_count` sucio del raw). Crea un ranking global de posts (excluyendo spam) ordenados por likes reales.

In [ ]:
# Tu solución aquí

---

<a id="ejercicio-33"></a>
### Ejercicio 33: Hashtag trends
Explota los hashtags de cada post (no spam), extrae la fecha, y agrupa por `(date, hashtag)` contando posts y usuarios únicos.

In [ ]:
# Tu solución aquí

---

<a id="ejercicio-34"></a>
### Ejercicio 34: Ranking diario de hashtags
Usando el resultado anterior, añade un ranking por día: para cada fecha, el hashtag con más posts es #1, el siguiente #2, etc. Usa una Window con `partitionBy("date")`.

In [ ]:
# Tu solución aquí

---

<a id="ejercicio-35"></a>
### Ejercicio 35: Features para detección de spam
Construye una tabla de features por usuario para un futuro modelo de spam. Incluye:
- `posts_count`, `posts_per_day`
- `avg_text_length` (longitud media del texto de sus posts)
- `unique_texts_ratio` (textos únicos / total posts)
- `avg_hashtags_per_post` (media de hashtags por post)
- `likes_received`, `likes_given`, `followers_count`, `following_count`
- `follow_ratio` = following / followers (si followers = 0, usa following directamente)
- `label` = `is_spam` convertido a entero (0/1)

In [ ]:
# Tu solución aquí

---

<a id="ejercicio-36"></a>
### Ejercicio 36: Aristas de interés compartido
Crea una tabla de aristas entre usuarios que comparten hashtags. Para cada par de usuarios (A, B) donde A < B (evitar duplicados), cuenta cuántos hashtags tienen en común. Esto es un **self-join** sobre la tabla de hashtags por usuario.

> **Pista:** Primero crea un DataFrame `(user_id, hashtag)` con valores distintos (explode + distinct). Luego haz self-join por hashtag con la condición `a.user_id < b.user_id`.

In [ ]:
# Tu solución aquí

---

# Bloque 5: Análisis final

Con los datos limpios y agregados, responde a estas preguntas analíticas sobre SocialLab.

---

<a id="ejercicio-37"></a>
### Ejercicio 37: Top 10 usuarios con más engagement
Muestra los 10 usuarios no-spam con mayor `avg_likes_per_post` (que tengan al menos 5 posts).

In [ ]:
# Tu solución aquí

---

<a id="ejercicio-38"></a>
### Ejercicio 38: Proporción de spam
¿Qué porcentaje de usuarios son spam? ¿Y qué porcentaje de posts? ¿Los usuarios spam generan más posts per cápita que los legítimos?

In [ ]:
# Tu solución aquí

---

<a id="ejercicio-39"></a>
### Ejercicio 39: Top 5 hashtags de toda la historia
Agrupa los hashtag trends y muestra los 5 hashtags con más posts totales (suma de `post_count` en todas las fechas).

In [ ]:
# Tu solución aquí

---

<a id="ejercicio-40"></a>
### Ejercicio 40: Análisis del grafo social
Combina las aristas FOLLOWS (de la tabla `follows`) con las aristas de SHARED_INTEREST (ejercicio 36). ¿Cuántas aristas hay de cada tipo? ¿Cuál tiene mayor peso medio?

In [ ]:
# Tu solución aquí

---

## ¡Práctica completada! 🎉

Has construido un pipeline ETL completo:

1. **Raw → Silver**: Limpieza de encoding, normalización de timestamps (5 formatos), deduplicación, eliminación de huérfanos, detección de spam
2. **Silver → Gold**: Métricas de usuario, rankings de engagement, trending hashtags, features para ML, grafo social

Este es el mismo pipeline que utiliza **SocialLab** internamente con PySpark.